# Notebook 4 — Structured Streaming
## E-REDES Big Data Analysis

Simulates a live smart-meter feed by replaying the test portion of the dataset as daily CSV files.
Spark Structured Streaming reads one file per trigger, applies the pre-fitted `lr_pipeline`,
and writes row-level predictions to Parquet while aggregating a live hourly demand summary.

**Prerequisites:** `Notebook3_MachineLearning.ipynb` must have run and saved `models/lr_pipeline`.

---


## 0. Environment Setup

In [6]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Running in Google Colab")

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/e_redes_v2/e_redes"
    )

    sys.path.append(str(PROJECT_ROOT))

except ImportError:
    print("Not running in Google Colab")

    PROJECT_ROOT = Path.cwd().parent
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.append(str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Google Colab


In [7]:
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-17-jdk-headless -qq
!java -version

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)


## 1. Imports

In [8]:
import shutil
import time
import warnings

warnings.filterwarnings('ignore')

import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType
)
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator

sys.path.append("/content/drive/MyDrive/e_redes/")
from utils.feature_eng import (
    TemporalFeatureTransformer,
    HolidayFlagTransformer,
    LagFeatureTransformer,
    NullLagDropTransformer,
)

print(f'PySpark {pyspark.__version__} — imports OK')

PySpark 4.0.2 — imports OK


## 2. SparkSession

> `spark.sql.shuffle.partitions=16` is intentionally low for a local Colab environment.
> In a production cluster set this to 2–3× the number of executor cores.

In [9]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('bda_eredes_streaming')
    .config('spark.executor.memory', '4g')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '16')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready')

Spark 4.0.2 ready


## 3. Paths & Configuration

In [10]:
from config import (
    STREAM_INPUT_DIR,
    STREAM_OUTPUT_DIR,
    CHECKPOINT_DIR,
    STREAM_SPLIT_DATE,
    MODEL_LR_PIPELINE_DIR,
    EREDES_PARQUET
)

STREAM_INPUT_DIR.mkdir(parents=True, exist_ok=True)
STREAM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

STREAM_INPUT_DIR = str(STREAM_INPUT_DIR)
STREAM_OUTPUT_DIR = str(STREAM_OUTPUT_DIR)
CHECKPOINT_DIR = str(CHECKPOINT_DIR)

print("Paths configured:")
for k, v in {
    "EREDES_PARQUET": EREDES_PARQUET,
    "MODEL_PIPELINE_DIR": MODEL_LR_PIPELINE_DIR,
    "STREAM_INPUT_DIR": STREAM_INPUT_DIR,
    "STREAM_OUTPUT_DIR": STREAM_OUTPUT_DIR,
    "CHECKPOINT_DIR": CHECKPOINT_DIR,
    "STREAM_SPLIT_DATE": STREAM_SPLIT_DATE,
}.items():
    print(f"  {k:<22}: {v}")

Paths configured:
  EREDES_PARQUET        : /content/drive/MyDrive/e_redes_v2/e_redes/data/processed/eredes_clean.parquet
  MODEL_PIPELINE_DIR    : /content/drive/MyDrive/e_redes_v2/e_redes/models/lr_pipeline
  STREAM_INPUT_DIR      : /content/drive/MyDrive/e_redes_v2/e_redes/data/stream/input
  STREAM_OUTPUT_DIR     : /content/drive/MyDrive/e_redes_v2/e_redes/data/stream/output
  CHECKPOINT_DIR        : /content/drive/MyDrive/e_redes_v2/e_redes/data/stream/checkpoints
  STREAM_SPLIT_DATE     : 2023-09-16


## 4. Declare the Stream Schema

> **Never use `inferSchema=True` in `readStream`.** Schema inference triggers a full
> scan of every arriving file — prohibitively expensive in a continuous stream.
> The schema must always be declared explicitly.

The schema here matches exactly what the daily CSV files will contain after the
feature transformers are applied in Section 5.

In [11]:
# This schema must match the columns written by the daily file preparation step (Section 5).
# It includes the raw columns PLUS all features added by the transformer chain.
stream_schema = StructType([
    # Raw columns
    StructField('municipality',            StringType(),    True),
    StructField('datetime',                TimestampType(), True),
    StructField('total_active_energy_kwh', DoubleType(),    True),
    StructField('year_month',              StringType(),    True),

    # Temporal features
    StructField('hour_of_day',             DoubleType(),    True),
    StructField('day_of_week',             DoubleType(),    True),
    StructField('is_weekend',              DoubleType(),    True),
    StructField('month',                   DoubleType(),    True),
    StructField('day_of_month',            DoubleType(),    True),

    # Holiday flag
    StructField('is_holiday',              DoubleType(),    True),

    # Lag features (ALL must be included)
    StructField('lag_168h',                DoubleType(),    True),
    StructField('lag_24h',                 DoubleType(),    True),
])

print(f'Schema: {len(stream_schema.fields)} fields')
for f in stream_schema.fields:
    print(f'  {f.name:<30} {str(f.dataType)}')

Schema: 12 fields
  municipality                   StringType()
  datetime                       TimestampType()
  total_active_energy_kwh        DoubleType()
  year_month                     StringType()
  hour_of_day                    DoubleType()
  day_of_week                    DoubleType()
  is_weekend                     DoubleType()
  month                          DoubleType()
  day_of_month                   DoubleType()
  is_holiday                     DoubleType()
  lag_168h                       DoubleType()
  lag_24h                        DoubleType()


## 5. Prepare Daily Stream Files

### Why this cell must run BEFORE `readStream` is defined

Spark's `FileStreamSource` scans the input directory at the moment `readStream` is
evaluated to build its initial offset log. If the directory is empty at that point,
the source records no files and processes 0 rows forever — even after files appear.

**Correct order:**
```
1. Write all daily files to STREAM_INPUT_DIR   ← Section 5
2. Define readStream                            ← Section 7
3. Start queries                               ← Section 8
```

### Files must be FLAT — no subdirectories

This was the **bug causing 0 rows** in the last setup. Spark's `FileStreamSource` does
**not recurse into subdirectories**. Writing `stream/input/2023-09-20/part-0.csv`
means Spark sees the folder `2023-09-20/`, ignores it (not a `.csv`), and reads nothing.

Fixed: write files as `stream/input/2023-09-20.csv` — flat, at the top level.
Since Spark always writes to a directory (not a single file), we use a temp subdir
and then `shutil.move` the single `part-*.csv` out to a flat named file.

In [12]:
# Reset directories / Create Directories
for d in [STREAM_INPUT_DIR, STREAM_OUTPUT_DIR, CHECKPOINT_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d)
print('Directories reset and created.')

Directories reset and created.


In [13]:
# Load and transform the stream source data
# Read from the cleaned Parquet (output of Notebook 1 ETL)
df_base = spark.read.parquet(EREDES_PARQUET)

# Isolate the test / stream window (everything after the split date)
# Filter uses a date predicate — Parquet predicate push-down applies here,
#   so only the relevant row groups are read from disk. Big-data-safe.
stream_source_df = df_base.filter(F.col('datetime') >= STREAM_SPLIT_DATE)

print(f'Stream source rows before feature engineering: {stream_source_df.count():,}')
print(f'Date range: {stream_source_df.agg(F.min("datetime"), F.max("datetime")).collect()[0]}')

Stream source rows before feature engineering: 99,802
Date range: Row(min(datetime)=datetime.datetime(2023, 9, 16, 0, 0), max(datetime)=datetime.datetime(2023, 9, 30, 22, 0))


In [14]:
# Apply the same feature transformer chain used in training
# These transformers are applied to the full stream source BEFORE splitting into
# daily files so that lag features at day boundaries are computed correctly.


#  feature note: lag_168h for the first day of the stream window
# ('16-09-2023') uses observations from '09-09-2023' (week before), which is in the training set.
# This is NOT data leakage — those values represent past observations that would
# genuinely be available at inference time in production.

stream_source_df = TemporalFeatureTransformer().transform(stream_source_df)
stream_source_df = HolidayFlagTransformer().transform(stream_source_df)
stream_source_df = LagFeatureTransformer().transform(stream_source_df)
stream_source_df = NullLagDropTransformer().transform(stream_source_df)

print(f'Rows after feature engineering + null lag drop: {stream_source_df.count():,}')
print(f'Columns: {stream_source_df.columns}')

Rows after feature engineering + null lag drop: 53,098
Columns: ['municipality', 'datetime', 'total_active_energy_kwh', 'year_month', 'hour_of_day', 'day_of_week', 'is_weekend', 'month', 'day_of_month', 'is_holiday', 'lag_168h', 'lag_24h']


In [15]:
# Write one CSV file per day FLAT into the stream input directory

# Files must be written as FLAT CSV files directly inside
# STREAM_INPUT_DIR, NOT inside dated subdirectories.

# The previous version wrote:
#   stream/input/2023-09-20/part-00000.csv   ← inside a subdir

# Spark's FileStreamSource does NOT recurse into subdirectories by default.
# It only sees files at the top level of the path. Subdirectories are silently
# ignored, so numInputRows was always 0.

dates = [
    row['date'] for row in
    stream_source_df
    .select(F.to_date('datetime').alias('date'))
    .distinct()
    .orderBy('date')   # ⚠ global sort acceptable — ~100 date values
    .collect()
]

print(f'Writing {len(dates)} daily CSV files flat into {STREAM_INPUT_DIR}...')

for day in dates:
    day_str = str(day)
    tmp_dir = f'{STREAM_INPUT_DIR}_tmp_{day_str}'   # temp dir for Spark to write into
    target  = f'{STREAM_INPUT_DIR}/{day_str}.csv'   # final flat file path

    # Write to a temp subdir (Spark always writes to a directory, not a file)
    (
        stream_source_df
        .filter(F.to_date('datetime') == day_str)
        .coalesce(1)   # single partition → single part file
        .write
        .mode('overwrite')
        .option('header', True)
        .csv(tmp_dir)
    )

    # Move the single csv file out of the temp dir to a flat named file
    import glob
    part_files = glob.glob(f'{tmp_dir}/part-*.csv')
    if part_files:
        shutil.move(part_files[0], target)
    shutil.rmtree(tmp_dir)   # clean up temp dir and _SUCCESS file

flat_files = list(Path(STREAM_INPUT_DIR).glob('*.csv'))
print(f'Done. {len(flat_files)} flat CSV files in {STREAM_INPUT_DIR}')
print('Sample:', sorted(f.name for f in flat_files)[:5])

Writing 8 daily CSV files flat into /content/drive/MyDrive/e_redes_v2/e_redes/data/stream/input...
Done. 8 flat CSV files in /content/drive/MyDrive/e_redes_v2/e_redes/data/stream/input
Sample: ['2023-09-23.csv', '2023-09-24.csv', '2023-09-25.csv', '2023-09-26.csv', '2023-09-27.csv']


## 6. Load Pre-Trained Pipeline

> The pipeline is loaded as a **read-only inference artifact**. It is never re-fitted
> inside the stream. Calling `.fit()` inside a streaming query is not supported by Spark
> and would be computationally infeasible — `fit()` requires a full dataset scan which
> would block the stream indefinitely.

In [16]:
loaded_pipeline = PipelineModel.load(MODEL_LR_PIPELINE_DIR)

print(f'Pipeline stages: {len(loaded_pipeline.stages)}')
for i, stage in enumerate(loaded_pipeline.stages):
    print(f'  Stage {i}: {type(stage).__name__}')

Pipeline stages: 4
  Stage 0: StringIndexerModel
  Stage 1: VectorAssembler
  Stage 2: StandardScalerModel
  Stage 3: LinearRegressionModel


## 7. Define the Streaming DataFrame

> `readStream` is defined HERE — **after** the daily files have been written in
> Section 5. This is the fix for the core bug. Spark now scans a populated directory
> and correctly registers all existing files in its offset log.

> The **watermark is mandatory**. Without it Spark maintains state for every
> event-time window it has ever seen, causing unbounded memory growth. The 2-hour
> watermark allows Spark to safely discard state for windows closed more than 2
> hours behind the latest observed event.

In [17]:
#  Read stream — defined AFTER files exist in the directory
# Schema is always explicit — inferSchema is never used in readStream
raw_stream = (
    spark.readStream
    .schema(stream_schema)
    .option('header', True)
    .option('maxFilesPerTrigger', 1)   # one day per trigger
    .option('latestFirst', False)      # chronological order
    .csv(STREAM_INPUT_DIR)
)

# Apply watermark
# Must be applied before any event-time window aggregation
watermarked_stream = raw_stream.withWatermark('datetime', '2 hours')

print(f'isStreaming: {raw_stream.isStreaming}')   # must be True

isStreaming: True


In [18]:
#  Apply pipeline (transform only — no fitting)
prediction_stream = loaded_pipeline.transform(watermarked_stream)

print(f'Prediction stream isStreaming: {prediction_stream.isStreaming}')

new_cols = set(prediction_stream.columns) - set(watermarked_stream.columns)
print(f'Columns added by pipeline: {sorted(new_cols)}')

Prediction stream isStreaming: True
Columns added by pipeline: ['features', 'features_raw', 'municipality_idx', 'prediction']


In [19]:
#  Row-level output — predictions alongside actuals
output_stream = prediction_stream.select(
    'municipality',
    'datetime',
    F.round('total_active_energy_kwh', 3).alias('actual_kwh'),
    F.round('prediction', 3).alias('predicted_kwh'),
    F.round(
        F.abs(F.col('total_active_energy_kwh') - F.col('prediction')), 3
    ).alias('abs_error_kwh'),
)

# Windowed aggregation — hourly demand summary
# outputMode must be 'update' (not 'append') for windowed aggregations
#   because window results can be revised as late data arrives within the watermark
hourly_demand_stream = (
    prediction_stream
    .groupBy(F.window('datetime', '1 hour').alias('window'))
    .agg(
        F.round(F.sum('prediction'),  2).alias('total_predicted_kwh'),
        F.round(F.avg('prediction'),  4).alias('avg_predicted_kwh'),
        F.round(F.max('prediction'),  4).alias('peak_predicted_kwh'),
        F.count('prediction')          .alias('meter_readings'),
    )
    .select(
        F.col('window.start').alias('window_start'),
        F.col('window.end').alias('window_end'),
        'total_predicted_kwh', 'avg_predicted_kwh',
        'peak_predicted_kwh', 'meter_readings',
    )
)

print(f'output_stream isStreaming       : {output_stream.isStreaming}')
print(f'hourly_demand_stream isStreaming : {hourly_demand_stream.isStreaming}')

output_stream isStreaming       : True
hourly_demand_stream isStreaming : True


## 8. Start Streaming Queries

> **Two queries, two checkpoint directories.** Each `writeStream` query must have
> its own dedicated checkpoint path. Sharing a checkpoint directory corrupts the
> offset log and causes incorrect or duplicated processing.

> **`awaitTermination(timeout)` is mandatory in a notebook.** If only `.start()` is non-blocking — it launches
> the query in a background thread and returns immediately. Without a blocking
> call afterwards, the notebook cell finishes before any trigger has fired,
> so `lastProgress` shows 0 rows. `awaitTermination(timeout=N)` blocks the
> cell for N seconds while triggers fire in the background.

In [20]:
#Query 1: Row-level predictions → Parquet sink
# append mode: each micro-batch appends new rows; no aggregation state
query_predictions = (
    output_stream
    .writeStream
    .outputMode('append')
    .format('parquet')
    .option('path', STREAM_OUTPUT_DIR)
    .option('checkpointLocation', f'{CHECKPOINT_DIR}/predictions')  # unique dir
    .trigger(processingTime='5 seconds')
    .queryName('predictions')
    .start()
)

# ── Query 2: Hourly aggregation → console sink ──────────────────────────────
# update mode: only updated window rows emitted per trigger
query_hourly = (
    hourly_demand_stream
    .writeStream
    .outputMode('update')
    .format('console')
    .option('numRows', 5)
    .option('truncate', False)
    .option('checkpointLocation', f'{CHECKPOINT_DIR}/hourly')       # unique dir
    .trigger(processingTime='5 seconds')
    .queryName('hourly_demand')
    .start()
)

print(f'Queries started: {[q.name for q in spark.streams.active]}')

Queries started: ['hourly_demand', 'predictions']


## 9. Wait for Batches & Monitor Progress

Block for `RUN_SECONDS` to allow multiple triggers to fire. Poll `lastProgress`
every 10 seconds to print a live summary.

> `awaitTermination(timeout=RUN_SECONDS)` is what actually allows rows to be
> processed. Without this block the notebook kernel would move to the next cell
> before a single trigger completes.

In [21]:
# How long to let the stream run.
# At 5s trigger + ~2s processing per batch, 90s processes ~20 daily files.
RUN_SECONDS = 90
POLL_EVERY  = 15    # print a progress line every N seconds

print(f'Streaming for {RUN_SECONDS}s ({RUN_SECONDS // 5} triggers max)...\n')

start_time = time.time()
while time.time() - start_time < RUN_SECONDS:
    time.sleep(POLL_EVERY)
    elapsed = int(time.time() - start_time)

    for q in spark.streams.active:
        prog = q.lastProgress
        if prog is None:
            print(f'  [{q.name:>18}]  no progress yet...')
            continue
        batch_id  = prog.get('batchId', '?')
        num_rows  = prog.get('numInputRows', 0)
        proc_rate = prog.get('processedRowsPerSecond', 0.0)
        sink_desc = prog.get('sink', {}).get('description', '')[:45]
        print(
            f'  [{q.name:>18}]  batch={batch_id:>3}  '
            f'rows={num_rows:>6,}  rate={proc_rate:>7.1f} rows/s  sink={sink_desc}'
        )

    print(f'  {" "*20}--- {elapsed}s elapsed ---')

print('\nTime limit reached — stopping queries...')
query_predictions.stop()
query_hourly.stop()

# Block until both queries have fully stopped
query_predictions.awaitTermination()
query_hourly.awaitTermination()
print('All queries stopped.')

Streaming for 90s (18 triggers max)...

  [     hourly_demand]  batch=  0  rows= 6,672  rate=  587.6 rows/s  sink=org.apache.spark.sql.execution.streaming.Cons
  [       predictions]  batch=  1  rows= 6,672  rate= 1070.3 rows/s  sink=FileSink[file:/content/drive/MyDrive/e_redes_
                      --- 15s elapsed ---
  [     hourly_demand]  batch=  3  rows= 6,672  rate= 1398.7 rows/s  sink=org.apache.spark.sql.execution.streaming.Cons
  [       predictions]  batch=  5  rows= 6,672  rate= 3786.6 rows/s  sink=FileSink[file:/content/drive/MyDrive/e_redes_
                      --- 30s elapsed ---
  [     hourly_demand]  batch=  6  rows= 6,672  rate= 1515.3 rows/s  sink=org.apache.spark.sql.execution.streaming.Cons
  [       predictions]  batch=  7  rows= 6,394  rate= 2487.0 rows/s  sink=FileSink[file:/content/drive/MyDrive/e_redes_
                      --- 45s elapsed ---
  [     hourly_demand]  batch=  8  rows=     0  rate=    0.0 rows/s  sink=org.apache.spark.sql.execution.streaming

## 10. Post-Stream Evaluation

Read the predictions written to Parquet back as a static DataFrame for metric
computation. This is the standard batch-over-stream evaluation pattern.

> `RegressionEvaluator` computes metrics as distributed aggregations — no
> `collect()` of raw rows to the driver. Big-data-safe.

In [22]:
predictions_df = spark.read.parquet(STREAM_OUTPUT_DIR)

n_total = predictions_df.count()
n_munis = predictions_df.select('municipality').distinct().count()
print(f'Predictions written : {n_total:,} rows across {n_munis} municipalities')
predictions_df.show(5, truncate=False)

Predictions written : 53,098 rows across 278 municipalities
+------------+-------------------+----------+-------------+-------------+
|municipality|datetime           |actual_kwh|predicted_kwh|abs_error_kwh|
+------------+-------------------+----------+-------------+-------------+
|abrantes    |2023-09-23 00:00:00|10378.859 |10084.514    |294.345      |
|abrantes    |2023-09-23 01:00:00|9608.829  |9175.891     |432.938      |
|abrantes    |2023-09-23 02:00:00|8826.028  |8475.312     |350.716      |
|abrantes    |2023-09-23 03:00:00|8679.696  |8141.667     |538.029      |
|abrantes    |2023-09-23 04:00:00|8339.741  |8339.832     |0.091        |
+------------+-------------------+----------+-------------+-------------+
only showing top 5 rows


In [23]:
rmse = RegressionEvaluator(
    labelCol='actual_kwh', predictionCol='predicted_kwh', metricName='rmse'
).evaluate(predictions_df)

mae = RegressionEvaluator(
    labelCol='actual_kwh', predictionCol='predicted_kwh', metricName='mae'
).evaluate(predictions_df)

r2 = RegressionEvaluator(
    labelCol='actual_kwh', predictionCol='predicted_kwh', metricName='r2'
).evaluate(predictions_df)

# MAPE computed as a distributed agg — .collect() only on a single scalar
mape = (
    predictions_df
    .filter(F.col('actual_kwh') > 0)
    .agg(
        F.avg(
            F.abs(F.col('actual_kwh') - F.col('predicted_kwh'))
            / F.col('actual_kwh') * 100
        ).alias('mape')
    )
    .collect()[0]['mape']  # single aggregated scalar — acceptable
)

print('═' * 42)
print('  Stream Inference — Evaluation Metrics')
print('═' * 42)
print(f'  Rows evaluated : {n_total:,}')
print(f'  RMSE           : {rmse:.4f} kWh')
print(f'  MAE            : {mae:.4f} kWh')
print(f'  R²             : {r2:.4f}')
print(f'  MAPE           : {mape:.2f} %')
print('═' * 42)

══════════════════════════════════════════
  Stream Inference — Evaluation Metrics
══════════════════════════════════════════
  Rows evaluated : 53,098
  RMSE           : 3755.8723 kWh
  MAE            : 1339.1605 kWh
  R²             : 0.9843
  MAPE           : 468.33 %
══════════════════════════════════════════


In [24]:
# Error breakdown by municipality (top 10 worst)
# groupBy + agg is fully distributed — big-data-safe
(
    predictions_df
    .filter(F.col('actual_kwh') > 0)
    .groupBy('municipality')
    .agg(
        F.count('*').alias('n'),
        F.round(F.avg('abs_error_kwh'), 3).alias('mean_abs_error'),
        F.round(
            F.avg(
                F.abs(F.col('actual_kwh') - F.col('predicted_kwh'))
                / F.col('actual_kwh') * 100
            ), 2
        ).alias('mape_pct')
    )
    # orderBy + limit: the limit bounds the data collected to driver — acceptable
    .orderBy(F.desc('mape_pct'))
    .limit(10)
    .show(truncate=False)
)

+-------------------+---+--------------+---------+
|municipality       |n  |mean_abs_error|mape_pct |
+-------------------+---+--------------+---------+
|vouzela            |191|487.439       |124519.76|
|mesão frio         |191|595.442       |143.89   |
|barrancos          |191|577.819       |128.39   |
|penedono           |191|583.706       |108.26   |
|sardoal            |191|596.167       |82.75    |
|manteigas          |191|593.295       |76.98    |
|mourão             |191|582.183       |72.8     |
|alfândega da fé    |191|581.64        |72.37    |
|gavião             |191|589.227       |71.81    |
|pampilhosa da serra|191|719.055       |71.12    |
+-------------------+---+--------------+---------+



## 11. Resource Cleanup

> Always stop streaming queries and clear cached RDDs before the notebook ends.
> Spark does not automatically free these when Python variables go out of scope.

In [25]:
for q in spark.streams.active:
    print(f'Stopping lingering query: {q.name}')
    q.stop()

if not spark.streams.active:
    print('No active streaming queries — clean.')

cached = spark.sparkContext._jsc.getPersistentRDDs()
for rdd_id, rdd in cached.items():
    rdd.unpersist()
print(f'Unpersisted {len(cached)} cached RDD(s).' if cached else 'No cached RDDs — memory clean.')

print('\nNotebook complete.')

No active streaming queries — clean.
No cached RDDs — memory clean.

Notebook complete.
